In [1]:
# analysis/analyze_tldr.ipynb
# Author: Bartosz Mamro
import os
import sys
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import MT5Tokenizer

project_root = os.path.abspath("..")
sys.path.append(project_root)

from utils import DATA_PATH
tokenizer = MT5Tokenizer.from_pretrained("google/mt5-base")

/Users/denghaowen/Desktop/multilingual-social-summary/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://githu

In [2]:
def analyze_jsonl(path):
    input_lens, summary_lens = [], []
    with open(path, 'r') as f:
        for line in tqdm(f, desc=f"Processing {os.path.basename(path)}"):
            ex = json.loads(line)
            body = ex.get("content")
            summary = ex.get("summary")
            if body and summary:
                input_lens.append(len(tokenizer.tokenize(body)))
                summary_lens.append(len(tokenizer.tokenize(summary)))
    return input_lens, summary_lens

def summarize_lengths(lengths):
    if len(lengths) == 0:
        return {"min": 0, "max": 0, "mean": 0, "median": 0, "std": 0}
    arr = np.array(lengths)
    return {
        "min": int(np.min(arr)),
        "max": int(np.max(arr)),
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr))
    }

In [3]:
variants = ["base", "sent", "noun", "full", "val"]
results = []

for variant in variants:
    file_name = f"tldr_train_{variant}.jsonl" if variant != "val" else "tldr_val.jsonl"
    path = os.path.join(DATA_PATH, "tldr_split", file_name)

    input_lens, summary_lens = analyze_jsonl(path)
    input_stats = summarize_lengths(input_lens)
    summary_stats = summarize_lengths(summary_lens)

    results.append({
        "variant": variant,
        **{f"input_{k}": v for k, v in input_stats.items()},
        **{f"summary_{k}": v for k, v in summary_stats.items()}
    })

Processing tldr_train_base.jsonl: 100it [00:00, 2851.12it/s]
Processing tldr_train_sent.jsonl: 100it [00:00, 3184.52it/s]
Processing tldr_train_noun.jsonl: 100it [00:00, 3054.36it/s]
Processing tldr_train_full.jsonl: 100it [00:00, 277034.61it/s]
Processing tldr_val.jsonl: 10it [00:00, 2630.98it/s]


In [8]:
from tabulate import tabulate

df = pd.DataFrame(results)
df.set_index("variant", inplace=True)

selected = df[["input_mean", "input_std", "summary_mean", "summary_std"]].copy()
selected = selected.round(1)
selected.columns = ["Input Mean", "Input Std", "Summary Mean", "Summary Std"]
latex_table = tabulate(selected, headers="keys", tablefmt="latex_booktabs")

print(latex_table)


\begin{tabular}{lrrrr}
\toprule
 variant   &   Input Mean &   Input Std &   Summary Mean &   Summary Std \\
\midrule
 base      &        307.7 &       258.1 &           30.1 &          35   \\
 sent      &        307.7 &       258.1 &           30.1 &          35   \\
 noun      &        307.7 &       258.1 &           30.1 &          35   \\
 full      &          0   &         0   &            0   &           0   \\
 val       &        326.3 &       184.5 &           25.6 &          15.9 \\
\bottomrule
\end{tabular}
